# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/praneethilluru/testtt/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Task Type: Supervised Classification & Opportunity Scoring.

Why: Content and SEO teams operate with limited sprint bandwidth. Rather than relying on static rules or vanity word counts, we formulate this task as predicting which content URLs will experience rank decay (loss of top-10 visibility) over a forward 30-day window. The model outputs a calibrated probability score (0.0 to 1.0) to directly rank and prioritize pages for editorial refresh.

In [5]:
# Lane task type definition check
task_type = "Supervised Classification & Priority Scoring"
print(f"Configured Task: {task_type}")

Configured Task: Supervised Classification & Priority Scoring


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target Definition:
* Target Column: `decay_target` (Binary flag: 1 = decaying/needs intervention, 0 = stable/growing).
* Label Provenance: Derived from observed historical performance across trailing 90-day intervals. An asset is marked as decaying if its trend direction is categorized as 'declining' (reflecting loss of average position or sustained impression drop).
* Forward Proxy: Forward impression velocity ratio coupled with rolling average rank stability.

In [6]:
target_col = "decay_target"
positive_class = "declining"
print(f"Target: {target_col} | Positive Class: {positive_class}")

Target: decay_target | Positive Class: declining


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Primary Metric: ROC-AUC and Precision@Top-20%.

What number means 'good':
* ROC-AUC > 0.75: Confirms the model discriminates decaying pages significantly better than random guessing (0.50) or simple volume heuristics.
* Precision@Top-20% > 0.70: Since an editorial team can only realistically update the top 20% highest-priority flagged pages per sprint, at least 70% of those top-recommended pages must be genuine decay candidates.

In [7]:
# Success metric benchmarks
target_roc_auc = 0.75
target_precision_top20 = 0.70
print(f"Target Thresholds -> ROC-AUC: {target_roc_auc}, Precision@Top20%: {target_precision_top20}")

Target Thresholds -> ROC-AUC: 0.75, Precision@Top20%: 0.7


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

Unit of Analysis:
* One row = One unique published content page asset (`url_id`) evaluated over a historical performance window.
* Features include search volume, 90-day impressions, and word count.

In [8]:
import pandas as pd

# Load starter dataset directly via GitHub raw URL
url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

# Create the binary target column
df["decay_target"] = (df["trend_direction"] == "declining").astype(int)

print(f"Unit of analysis shape: {df.shape[0]} rows (pages) x {df.shape[1]} columns")

# Display the first few rows showing unit of analysis
display(df[["search_volume", "impressions_90d", "word_count", "trend_direction", "decay_target"]].head())

Unit of analysis shape: 30000 rows (pages) x 45 columns


,search_volume,impressions_90d,word_count,trend_direction,decay_target
0,10.0,3803,3221.0,down,0
1,90.0,15320,2481.0,down,0
2,0.0,12581,3515.0,down,0
3,10.0,11751,NaN,stable,0
4,0.0,19140,2803.0,down,0


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Why ML beats a fixed rule:
Fixed heuristic rules (such as "flag any article older than 6 months" or "flag pages with fewer than 1,500 words") consistently fail because content visibility signals interact non-linearly:
1. A concise, focused glossary entry with 350 words can stably hold position 1 for years without decaying.
2. A 3,500-word comprehensive guide can experience sharp traffic decay due to SERP layout changes, intent shifts, or cannibalization despite meeting traditional length guidelines.
3. A machine learning model detects multi-dimensional thresholds across volume velocity, search volume, and word count that static `if/else` rules cannot capture.

In [9]:
# Demonstrate non-linear separation: median word counts across trend directions
word_count_summary = df.groupby("trend_direction")["word_count"].median()
print("Median word count across trends:\n", word_count_summary)
print("\nObservation: Declining and growing pages share similar median word counts, showing fixed rules based on length fail.")

Median word count across trends:
 trend_direction
down      2909.0
flat      2698.5
new       2239.0
stable    2912.5
up        2847.5
Name: word_count, dtype: float64

Observation: Declining and growing pages share similar median word counts, showing fixed rules based on length fail.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.